In [ ]:
import sys, os
sys.path.append(os.getcwd() + "/..")

# Parallel Computation on GPUs

In [ ]:
import torch
from utils.benchmark import Benchmark

In [ ]:
device_count = torch.accelerator.device_count()
assert device_count > 1, "Must have multiple devices"
assert torch.cuda.is_available(), "Multiple cuda devices are required to run this tutorial"
devices = [torch.cuda.device(i) for i in range(device_count)]
print(devices)

In [ ]:
N = 50
M = 4000

In [ ]:
def run(x):
    return [x.mm(x) for _ in range(N)]

x_gpu0 = torch.randn(size=(M, M), device=devices[0])
x_gpu1 = torch.randn(size=(M, M), device=devices[1])

In [ ]:
# warm up devices
run(x_gpu0)
run(x_gpu1)
torch.cuda.synchronize(devices[0])
torch.cuda.synchronize(devices[1])

In [ ]:
with Benchmark('GPU 0 time'):
    run(x_gpu0)
    torch.cuda.synchronize(devices[0])

with Benchmark('GPU 1 time'):
    run(x_gpu1)
    torch.cuda.synchronize(devices[1])

In [ ]:
with Benchmark("GPU 0 & GPU 1"):
    run(x_gpu0)
    run(x_gpu1)
    torch.cuda.synchronize()

# Parallel Computation and Communication

In [ ]:
def copy_to_cpu(x, non_blocking=False):
    return [y.to('cpu', non_blocking=non_blocking) for y in x]

In [ ]:
with Benchmark('Run on GPU 0'):
    y = run(x_gpu0)
    torch.cuda.synchronize()

with Benchmark('Copy to CPU'):
    y_cpu = copy_to_cpu(y)
    torch.cuda.synchronize()

In [ ]:
with Benchmark('Run on GPU1 and asynchronously copy to CPU'):
    y = run(x_gpu0)
    y_cpu = copy_to_cpu(y, non_blocking=True)
    torch.cuda.synchronize()